# 🔧 Firmware — AetherNet IoT & Autonomous Rover
**Proyecto:** Plataforma distribuida domótica + rover tanque (Sprint 1-3) — 100% FOSS `docs/prd.md:53`  
**Stack:** `arduino-cli 1.5.1` + `ESP32-WROOM-32U` + `MEGA 2560` + `UNO R3` + `nRF24L01` + `Mosquitto` + `FastAPI`  
**Rama:** `feature/app-joystick-virtual` (MOV-05/06 RF-1.2 validado HW 2026-09-11)  
**Autor:** Andres Felipe Martinez Henao — 2a electrónica/Arduino, 1a C, 2a Python/JS/React  
**Logs:** `docs/logs/firmware_sprint3/` (T1 Docker, T2 MEGA, T3 Gateway, T4 Rover)

> **Cómo usar:** Lee markdown en orden. Ejecuta celdas `In [ ]` (Shift+Enter) para verificar archivos y parsear logs reales. Los firmware viven en `firmware/` (fuente verdad).


---
## 📑 Índice
1. [Mapa firmware — qué hay y dónde](#mapa)
2. [MEGA Access — cerrojo + láser (RF-2.2 HU-01/02)](#mega)
3. [Gateway ESP32 — puente MQTT↔UART↔RF (RF-2.1)](#gateway)
4. [Rover UNO — tanque autónomo (RF-3.1/3.2 HU-03/04)](#rover)
5. [Validación Sprint 3 — Joystick nRF24 extremo a extremo](#validacion)
6. [Logs guardados — 4 terminales 2026-09-11](#logs)
7. [Reproducibilidad — `arduino-cli` + Docker](#repro)
8. [Siguientes pasos](#next)


<a id="mapa"></a>
---
## 1. 🗺️ Mapa firmware — qué hay y dónde

```
firmware/
├── gateway-esp32/gateway-esp32.ino   26k  ESP32 puente WiFi+MQTT+UART+RF  RF-2.1 DEVOPS-05
│   ├── secrets.h          .gitignore  WIFI FELIPE./2516f751 192.168.1.14:1883/8000
│   └── secrets.h.example  plantilla CI
├── mega-access/           5.5k +src/  MEGA cerrojo + laser RF-2.2/2.3 HU-01/02
│   ├── mega-access.ino    orchestrador (setup/loop)
│   └── src/config.h door.h door.cpp keypad_control.h led.h laser.h uart_protocol.h
├── rover-uno/rover-uno.ino 23k        UNO L298N + HC-SR04 + TCRT×3 + nRF  RF-3.1/3.2 HU-03/04
├── test-nrf24-esp32 / test-nrf24-uno   bancos RF aislados
├── test-link-esp32-rover / test-link-uno-rover  ciclo TX→RX DEVOPS-05
└── test-laser-uno / test-ema-uno     bancos KY-008 + EMA α=0.2
```

| Módulo | FQBN | Pines críticos | Flash hoy |
|---|---|---|---|
| **Gateway ESP32** | `esp32:esp32:esp32` | `CE5 CSN15 SCK18 MOSI23 MISO19` `UART2 16/17@38400` `WiFi 2.4G` | 26k `T3 OK` |
| **MEGA** | `arduino:avr:mega` | `Keypad 22/24/26/28 30/32/34/36` `Servo9` `LED44/45/46` `Laser TX8 RX7` `UART Serial2` | `20358b 8%` `T2 OK` |
| **Rover UNO** | `arduino:avr:uno` | `ENA5 IN1 6 IN2 7 IN3 8 IN4 9 ENB11` `US TRIG2 ECHO3` `IR A0/A1/A2` `CE4 CSN10` | `6900b 21%` `T4 OK` |


In [ ]:
# Celda 1 — Verifica estructura real (ejecútala)
from pathlib import Path
import textwrap
base = Path("firmware")
if not base.exists():
    base = Path("../firmware")
if not base.exists():
    base = Path("/home/craos6518/Documentos/AetherNet-IoT-Autonomous-Rover/firmware")
print(f"base={base} existe={base.exists()}")
for p in sorted(base.rglob("*.ino")):
    print(f"{p.relative_to(base.parent)} — {p.stat().st_size/1024:.1f}kB")
print("\nArchivos clave:")
for f in ["gateway-esp32/secrets.h","mega-access/src/config.h","rover-uno/rover-uno.ino"]:
    pp = base/f if (base/f).exists() else base.parent/f
    # fallback
    if not (base/f).exists():
        pp = Path("firmware")/f
    print(f"  {f}: {'OK' if pp.exists() else 'NO'} {pp.stat().st_size if pp.exists() else 0} bytes")


<a id="mega"></a>
---
## 2. 🔐 MEGA Access — RF-2.2 (HU-01) + RF-2.3 (HU-02)

**Hardware:** `MEGA 2560` + `Keypad 4×4` `ROW 30/32/34/36 COL 22/24/26/28` + `Servo MG90S 9` `90° unlock 5s` + `LED RGB 44/45/46` + `Laser KY-008 TX8 → LDR 7 (divisor 10k)`  
**Lógica:** `src/keypad_control.cpp:51 isDigit + hashPin djb2` → `uart_protocol.cpp:40 CMD:ACCESS` → `gateway-esp32.ino:332 handleAccessCommand` → `POST /api/access-events 201` + `aethernet/access/event`  
**Laser:** `laser.cpp:32 laserInit / handleLaser 50ms CHECK / 3s COOLDOWN` → `SECURITY:{"event_type":"intrusion"}`

**Log T2 2026-09-11** (`docs/logs/firmware_sprint3/T2_mega_2026-09-11.log`):

```text
=== AetherNet MEGA Cerrojo RF-2.2/HU-01 ===
Laser KY-008 + LDR discreta init — TX:8 HIGH, RX:7 INPUT (divisor 10k), armed=true
MEGA Cerrojo+Laser listo — PIN 1234 | Auto-lock 5s | LED verde 5s / rojo 3s intrusión
[PIN DBG] buffer='1234' len=4 → Door UNLOCKED (90°) → [UART TX] ACCESS:{"pin_hash":"7c78c98f","success":true}
!!! INTRUSION DETECTED — LASER BREAK !!! → [UART TX] SECURITY:{"event_type":"intrusion","sensor":"laser-01"}
```

Validado: `1234# → 7c78c98f GRANTED` + `123# → b8789fb DENIED` + laser dispara `SECURITY` → Gateway `Backend POST 201` (`T3`).


In [ ]:
# Celda 2 — Inspecciona config MEGA real
import pathlib
p = pathlib.Path("firmware/mega-access/src/config.h")
if not p.exists(): p = pathlib.Path("/home/craos6518/Documentos/AetherNet-IoT-Autonomous-Rover/firmware/mega-access/src/config.h")
txt = p.read_text()[:1200]
print(txt)
# Busca PIN y LED
import re
for kw in ["VALID_PIN","DOOR_AUTO_LOCK","LED","LASER"]:
    for line in txt.splitlines():
        if kw.lower() in line.lower():
            print(line.strip())


<a id="gateway"></a>
---
## 3. 🌉 Gateway ESP32 — RF-2.1 (DEVOPS-05/11)

**Rol:** único con WiFi → traduce `MQTT aethernet/# ←→ UART MEGA ←→ nRF24 Rover` + `HTTP POST FastAPI`  
**Credenciales:** `secrets.h` (`.gitignore`) `WIFI FELIPE./2516f751` `MQTT 192.168.1.14:1883` `BACKEND 8000` — fallback `gateway-esp32.ino:59 #ifndef` para CI  
**Topics:** `aethernet/rover/command:69 (App→Rover)` `aethernet/rover/telemetry` `aethernet/access/event` `aethernet/mega/status` (ver `backend/mosquitto/config/acl.conf:14 aethernet/#`)  
**RF:** `RF24 CE5 CSN15` `channel 76 2MBPS PA_HIGH` `isChipConnected()` + `radio.printDetails()` `STATUS 0x0e` `RF_CH 0x4c` `PA_HIGH`

**Log T3 2026-09-11** (fragmento, completo en `docs/logs/firmware_sprint3/T3_gateway_2026-09-11_full.log`):

```text
=== AetherNet Gateway ESP32 Starting ===
UART to MEGA initialized / nRF24L01 initialized
WiFi connected: 192.168.1.23 / MQTT reconnecting...connected
[MEGA UART RX] STATUS:{"door_locked":true,"laser_armed":true,"free_ram":7238}
[MEGA UART RX] ACCESS:{"pin_hash":"7c78c98f","success":true} → Backend ACCESS POST ok (201)
MQTT RX: aethernet/rover/command -> {"left_pwm":120,"right_pwm":120,"mode":1}
RF TX: L=120 R=120 mode=1
```

**Incidencia inicial (resuelta):** `firmware/test-nrf24-esp32` (LINK TEST) estuvo flasheado → `TX FAIL/no ACK` `isChipConnected` intermitente (falta `C 10µF` en 3.3V). Reflasheado a `gateway-esp32.ino` → `MQTT RX→RF TX` estable.


In [ ]:
# Celda 3 — Parsea T3: cuenta MQTT RX vs RF TX (debe ser 1:1)
from pathlib import Path
import re
p = Path("docs/logs/firmware_sprint3/T3_gateway_2026-09-11_full.log")
if not p.exists():
    p = Path("/home/craos6518/Documentos/AetherNet-IoT-Autonomous-Rover/docs/logs/firmware_sprint3/T3_gateway_2026-09-11_full.log")
txt = p.read_text()
rx = len(re.findall(r"MQTT RX:", txt))
tx = len(re.findall(r"RF TX:", txt))
print(f"T3: MQTT RX={rx} RF TX={tx} ratio={tx/rx if rx else 0:.2f} (ideal 1.0)")
# Extrae últimos 5 comandos
lines = [l for l in txt.splitlines() if "MQTT RX" in l or "RF TX" in l]
for l in lines[-10:]:
    print(l)


<a id="rover"></a>
---
## 4. 🤖 Rover UNO — RF-3.1 (manual) + RF-3.2 (auto) + HU-03/04

**Hardware:** `UNO` `L298N ENA5 IN1 6 IN2 7 IN3 8 IN4 9 ENB11` (tanque diferencial TT 6V 1:48) + `HC-SR04 TRIG2 ECHO3 MAX200cm` + `TCRT5000 A0 trasero/A1 izq/A2 der THRESH 500 invertido >500 BORDE` + `nRF CE4 CSN10`  
**Firmware:** `rover-uno.ino:80 FAILSAFE 500ms` `MAX_PWM 255` `MIN_PWM 60` `EMA α=0.2` `rover-uno.ino:259` `analogRead >500 BORDE` pista blanca pot mitad 3mm (antes `<500` invertido 2026-09-11) — `test-rover-sensors.ino 4712b` banco `A0 854 OK negra / 290 BORDE blanca`  
**Modos:** `mode 0 stop (0,0)` `mode 1 manual (joystick)` `mode 2 auto (evasión US+IR)`  
**Fail-safe HU-04:** `if (millis()-lastValidPacketTime>500) stopMotors()` + `verifyChecksum()` `struct RoverCommand {int16 left,right; uint8 mode; uint8 checksum}`

**App Joystick** `feature/app-joystick-virtual`:
- `JoystickScreen.kt:1 Canvas 240dp detectDragGestures` → `JoystickMapper.normalizeInCircle 120dp → X,Y -1..1` (Y invertida)
- `JoystickMapper.kt:28 vectorToPwm (y±x)*255 clamp + deadband 60 espejo UNO`
- `JoystickViewModel.kt:47 throttle 50ms (20Hz) + force stop onReleased`
- `MqttManager.kt:215 publishRoverCommand QoS0 retain false` → `AetherRepositoryImpl.kt:116 sendRoverCommand`

**Log T4 2026-09-11** (`T4_rover_2026-09-11_full.log` + `live_2026-09-11/T_telemetry_invertida.log 89`):

```text
=== AetherNet Rover UNO Starting ===
nRF24L01 initialized / Rover ready - awaiting commands
!!! FAIL-SAFE ACTIVATED: No RF signal !!!  # sin TX, normal HU-04
RF RX: L=120 R=120 mode=1  # espejo T3
RF RX: L=195 R=0 mode=1   # giro derecha tank
RF RX: L=0 R=0 mode=0     # stop
```

**Validación sensores 2026-09-11 14:28:** `test-rover-sensors` `A0_Rear 854 OK` negra `vs 290 BORDE` blanca `>500 BORDE` `A1 607 OK` `A2 696 OK` + `HC-SR04 US_raw 6-33cm EMA 6-32cm <<< OBSTACULO <30cm` `OBSTACLE 30cm CRITICAL 15cm` + telemetría `aethernet/rover/telemetry {"ir_left":true/false,"ir_right":true/false,"ultrasonic_cm":0-95}` 89 líneas `T_telemetry_invertida.log` — **TCRT + US sí funcionan en telemetría** (ver §6 y celda 4b).


In [ ]:
# Celda 4 — Valida T3→T4 1:1 (comandos deben coincidir)
from pathlib import Path
import re
def extract(p):
    import pathlib
    path = pathlib.Path(p)
    if not path.exists():
        path = pathlib.Path("/home/craos6518/Documentos/AetherNet-IoT-Autonomous-Rover") / p
    txt = path.read_text()
    # L=.. R=.. mode=..
    return re.findall(r"L=(-?\d+) R=(-?\d+) mode=(\d+)", txt)

t3 = extract("docs/logs/firmware_sprint3/T3_gateway_2026-09-11_full.log")
t4 = extract("docs/logs/firmware_sprint3/T4_rover_2026-09-11_full.log")
print(f"T3 cmds={len(t3)} T4 cmds={len(t4)}")
# compara primeros N comunes
n = min(len(t3), len(t4))
match = sum(1 for a,b in zip(t3[:n], t4[:n]) if a==b)
print(f"Match primeros {n}: {match}/{n} ({match/n*100:.1f}%) {'✅ RF link OK' if match/n>0.95 else '❌ desync'}")
for i, (a,b) in enumerate(zip(t3[:6], t4[:6])):
    print(f"{i}: T3 {a} vs T4 {b} {'==' if a==b else '!='}")


<a id="validacion"></a>
---
## 5. ✅ Validación Sprint 3 — Joystick nRF24 extremo a extremo

**Fecha:** `2026-09-11` — `feature/app-joystick-virtual` + firmwares `gateway 26k / rover 6.9k / mega 20.3k`  
**Entorno:** `Docker 0.0.0.0:8000 health ok` `Mosquitto 1883/9001 aethernet/#` `FELIPE. 192.168.1.14/23` `App SM-X620 MOV-05/06`

| Paso | Esperado | Observado | Veredicto |
|---|---|---|---|
| `Docker T1` `docker compose up` | `postgres healthy 5432` `mosquitto 1883/9001` `fastapi 8000` | `Started` `health ok` `Uvicorn 0.0.0.0:8000` | ✅ |
| `mosquitto_sub aethernet/#` | `aethernet/rover/command` JSON | `{"left_pwm":120,"right_pwm":120,"mode":1}` rafaga 20Hz | ✅ `MOV-06` |
| `mosquitto_pub rover/command` | `T3 MQTT RX→RF TX` | `MQTT RX ... → RF TX: L=120 R=120` | ✅ `RF-2.1` |
| `Joystick App` drag → `Canvas X,Y -1..1` | `MqttManager publishRoverCommand:215` | `mosquitto_sub` vio `L 195 R0`, `L0 R-198`, `tank 255/-255` | ✅ `MOV-05` |
| `T4 RF RX` | `L/R -255..255 mode 0/1` espejo T3 | `RF RX: L=120 R=120` 1:1 con T3 (celda 4) | ✅ `RF-3.1` |
| `Fail-safe 500ms HU-04` | `stopMotors si >500ms sin RF` | `!!! FAIL-SAFE !!!` solo en gaps, se recupera al siguiente RX | ✅ |
| `MEGA→Gateway→Backend` | `ACCESS 7c78c98f 201` `SECURITY` | `Backend POST ok (201)` `aethernet/access/event` | ✅ `HU-01/02` |
| `Telemetría` | `aethernet/rover/telemetry` `rf_rssi -70` | `{"left_pwm":120,"rf_rssi":-70,"ir_*":true}` `rf_connected:true` | ✅ |
| `TCRT5000` `A0 tras/A1 izq/A2 der` `THRESH 500 >500 BORDE` | `analogRead >500` pot mitad 3mm | `test-rover-sensors` `A0 854 OK negra 290 BORDE blanca` + `telemetry ir_left true/false ir_right true/false` 89 líneas | ✅ `RF-3.2` pista blanca |
| `HC-SR04` `TRIG2 ECHO3 EMA α=0.2` `OBSTACLE 30cm` | `US_raw 6-33cm EMA 6-32cm` | `test-rover-sensors` `US_raw 6cm EMA 6cm <<< OBSTACULO <30cm` + `telemetry ultrasonic_cm 0-95 EMA` | ✅ `HU-03` |

**Latencia:** `JoystickViewModel THROTTLE 50ms` `JoystickMapper MAX 255` `Gateway RF <10ms 2MBPS` cumple `prd.md:49 <50ms MQTT / 0:50 <10ms RF`.

**Diagrama flujo validado:**

```mermaid
flowchart LR
  App[JoystickScreen Canvas 120dp<br/>X,Y -1..1] -->|vectorToPwm throttle 50ms| VM[JoystickViewModel]
  VM -->|sendRoverCommand 255/-255| Mqtt[MqttManager QoS0<br/>aethernet/rover/command]
  Mqtt -->|WiFi FELIPE.| GW[Gateway ESP32<br/>CE5 CSN15 Ch76 2MBPS]
  GW -->|radio.write RoverCommand| Rover[UNO CE4 CSN10<br/>L298N ENA5/ENB11]
  Rover -->|telemetry HC-SR04 EMA α=0.2| GW2[Gateway handleRfCommunication]
  GW2 -->|aethernet/rover/telemetry| App2[Dashboard MQTT ●]
  MEGA[MEGA Keypad/Laser] -->|UART 38400| GW
  GW -->|POST /api/access-events| Backend[(FastAPI PG)]
```


In [ ]:
# Celda 5 — Latencia estimada y deadband check
# Simula JoystickMapper deadband 60 idéntico a rover-uno.ino:91
def vector_to_pwm(x,y, MAX=255, MIN=60):
    l = int((y+x)*MAX)
    r = int((y-x)*MAX)
    l = max(-MAX, min(MAX, l))
    r = max(-MAX, min(MAX, r))
    if abs(l) < MIN: l=0
    if abs(r) < MIN: r=0
    return l,r

tests = [(0,0.1),(0.1,0.1),(0,1),(1,0),(0.5,0.5)]
for x,y in tests:
    print(f"x={x} y={y} -> L,R={vector_to_pwm(x,y)}")
print("Deadband 60 bloquea (0,0.1) cerca centro — evita zumbido oruga (medido 40 no vence fricción)")


<a id="logs"></a>
---
## 6. 📂 Logs guardados — `docs/logs/firmware_sprint3/`

> Logs completos del **2026-09-11** — 4 terminales en paralelo. Copia/pega directo de `arduino-cli monitor` + `docker compose` + `mosquitto_sub`. Si necesitas los `.log` de **todos los firmware** (test-nrf24, test-laser, test-ema) avisa y los añado aquí + `docs/logs/`.

| Terminal | Archivo | Contenido | Líneas |
|---|---|---|---|
| **T1 Docker** | `T1_docker_2026-09-11.log` | `compose up` `health ok` `mosquitto_sub aethernet/#` ráfaga joystick | ~35 |
| **T2 MEGA** | `T2_mega_2026-09-11.log` | `Laser init` `PIN 1234 7c78c98f GRANTED` `SECURITY intrusion` | ~22 |
| **T3 Gateway** | `T3_gateway_2026-09-11.log` + `_full.log` | `WiFi 192.168.1.23 MQTT connected` `MQTT RX→RF TX L/R mode` | 45+ |
| **T4 Rover** | `T4_rover_2026-09-11.log` + `_full.log` | `nRF initialized` `RF RX L/R mode` `FAIL-SAFE 500ms` | 40+ |
| **T2A MQTT** | `mosquitto_sub aethernet/#` | `aethernet/mega/status` `aethernet/rover/telemetry rf_rssi -70` | 120+ |
| **T4 Sensores** | `test-rover-sensors/test-rover-sensors.ino` 4712b | `A0 854 OK negra 290 BORDE blanca` `A1 607 OK` `A2 696 OK` `US 6-33cm EMA` `<<< OBSTACULO` | 89 telemetría |
| **T5 Telemetría** | `live_2026-09-11/T_telemetry_invertida.log` | `aethernet/rover/telemetry` `ir_left true/false` `ultrasonic 0-95` `rf_rssi -70` 89 líneas — **sensores sí funcionan en telemetría** | 89 |

| **Test nRF24** | `firmware/test-nrf24-esp32` + `test-nrf24-uno` / `test-link-*` | `RF24 CE5/CSN15 vs CE4/CSN10 Ch76 2MBPS printDetails STATUS 0x0e` | — |
| **Test Laser** | `firmware/test-laser-uno` | `KY-008 S→8 + LDR 7 divisor 10k TCRT threshold 500 analogRead` | — |
| **Test EMA** | `firmware/test-ema-uno` | `EMA α=0.2 S_t=α·Y_t+(1-α)·S_{t-1} D2/D3 filtro stats/ema_filter.py:15` | — |
| **Telemetría invertida** | `docs/logs/firmware_sprint3/live_2026-09-11/T_telemetry_invertida.log` | `aethernet/rover/telemetry ir_left/ir_right ultrasonic 0-95 rf_rssi -70` | 89 |
**Ubicación:** `docs/logs/firmware_sprint3/` — añade al `.gitignore` si son pesados, pero este Sprint los versionamos para auditoría `sprints.md:60`.

**Acceso desde Python:**

```python
from pathlib import Path
for p in Path("docs/logs/firmware_sprint3").glob("*.log"):
    print(p, len(p.read_text().splitlines()), "líneas")
```


In [ ]:
# Celda 6 — Lista logs guardados
from pathlib import Path
base = Path("docs/logs/firmware_sprint3")
if not base.exists():
    base = Path("/home/craos6518/Documentos/AetherNet-IoT-Autonomous-Rover/docs/logs/firmware_sprint3")
for p in sorted(base.glob("*.log")):
    lines = p.read_text().splitlines()
    print(f"{p.name} — {len(lines)} líneas — {p.stat().st_size} bytes — preview: {lines[0][:70]}")


**Update 2026-09-11 14:28 — TCRT/HC-SR04 telemetría invertida (ya integrado en §4-6):**

* **Pista blanca final** pot mitad 3mm: `A0 854 OK` negra vs `A0 290 BORDE` blanca — lógica invertida `analogRead >500 BORDE` validada `test-rover-sensors 4712b` + `T_telemetry_invertida.log 89` `ir_left true/false ir_right true/false` `ultrasonic 0-95cm EMA α=0.2` `ir_center` trasero.
* **HC-SR04 EMA** `TRIG2 ECHO3` `US_raw 6-33cm EMA 6-32cm <<< OBSTACULO <30cm` estable con luz/oscura, `OBSTACLE 30cm CRITICAL 15cm` `executeAutoMode:379`.
* Ver `docs/logs/firmware_sprint3/live_2026-09-11/T_telemetry_invertida.log` 89 líneas 14:28.


## 📸 Fotos hardware — placeholders

> Checklist fotos pendientes: ver `docs/Firmware/README.md` checklist — si no existen deja `![FOTO PENDIENTE]`.

![FOTO PENDIENTE](../../docs/Firmware/fotos/mega-panel.jpg) — tomar foto cenital panel MEGA con regla, 12MP
![FOTO PENDIENTE](../../docs/Firmware/fotos/gateway-esp32.jpg) — ESP32 + nRF CE5 CSN15
![FOTO PENDIENTE](../../docs/Hardware/fotos/rover-uno.jpg) — UNO + L298N + TCRT
![FOTO PENDIENTE](../../docs/Hardware/fotos/chasis-tt.jpg) — TT 6V 1:48 ×4

> Referencia: `docs/Firmware/README.md` — checklist fotos MEGA/Gateway/Rover/Chasis.


In [ ]:
# Fotos firmware — verifica pendientes
from pathlib import Path
for p in Path("docs/Firmware/fotos").glob("*"):
    print(p, "OK", p.stat().st_size)
if not list(Path("docs/Firmware/fotos").glob("*.jpg")):
    print("FOTOS PENDIENTES — ver docs/Firmware/README.md checklist")
for p in Path("docs/Hardware/fotos").glob("*"):
    print(p, "OK")


<a id="repro"></a>
---
## 7. 🔁 Reproducibilidad — `arduino-cli` + Docker

**Requisitos:** `arduino-cli 1.5.1` + cores `arduino:avr` + `esp32:esp32` + `Docker` + `Mosquitto clients`

```bash
# T1 Docker
docker compose up --build -d && curl http://192.168.1.14:8000/health  # {"status":"ok"}
mosquitto_sub -h 192.168.1.14 -t "aethernet/#" -v   # deja abierto

# T2 MEGA (HU-01/02)
arduino-cli compile --fqbn arduino:avr:mega ./firmware/mega-access  # 20358b 8%
arduino-cli upload -p /dev/ttyACM0 --fqbn arduino:avr:mega ./firmware/mega-access
arduino-cli monitor -p /dev/ttyACM0 -c baudrate=115200  # 1234# → GRANTED

# T3 Gateway (RF-2.1)  — usa secrets.h real, no test-nrf24!
cat firmware/gateway-esp32/secrets.h  # FELIPE./2516f751 192.168.1.14
arduino-cli compile --fqbn esp32:esp32:esp32 ./firmware/gateway-esp32
arduino-cli upload -p /dev/ttyUSB0 --fqbn esp32:esp32:esp32 ./firmware/gateway-esp32
arduino-cli monitor -p /dev/ttyUSB0 -c baudrate=115200  # WiFi 192.168.1.23 MQTT connected

# T4 Rover (RF-3.1)
arduino-cli compile --fqbn arduino:avr:uno ./firmware/rover-uno  # 6900b 21%
arduino-cli upload -p /dev/ttyACM1 --fqbn arduino:avr:uno ./firmware/rover-uno
arduino-cli monitor -p /dev/ttyACM1 -c baudrate=115200  # RF RX: L=120...

# Test sin App
mosquitto_pub -h 192.168.1.14 -t "aethernet/rover/command" -m '{"left_pwm":120,"right_pwm":120,"mode":1}'
# → T3 MQTT RX → RF TX: L=120 → T4 RF RX: L=120

# App Joystick (MOV-05/06)
./gradlew :app:assembleDebug && ./gradlew :app:installDebug  # o Android Studio
# Dashboard → Joystick Rover → drag 120dp → T3/T4 espejo
```

**CI:** `.github/workflows/ci.yml:92 matrix firmware: gateway-esp32/esp32:esp32:esp32 mega-access/arduino:avr:mega rover-uno/arduino:avr:uno`

**Capacitor nRF:** `ESP32 10µF 3.3V-GND` + `MEGA C1/C2 10µF` (ya en `sprints.md:13` fix `CSN 18→15`) evita `isChipConnected=0`.


<a id="next"></a>
---
## 8. 🚀 Siguientes pasos

| Sprint | Tarea | Estado | Depende de |
|---|---|---|---|
| **3** | `MOV-05/06 Joystick` | ✅ validado 2026-09-11 | RF DEVOPS-05 |
| **3** | `MOV-07 Fallback BT SPP RF-1.3` | 🔜 usa HC-06 MEGA `RF-1.3` | este notebook |
| **4** | `MOV-08 Dashboard consolidado` | 🔜 integra `LedStatusCard + MQTT ● + Rover Card` | MOV-02/03 |
| **4** | `MOV-10 Tests JUnit` | ✅ `JoystickMapperTest 10` `JoystickViewModelTest 6` | — |
| **Extra** | `Logs todos los firmware` | ❓ pide si los quieres | `test-laser-uno`, `test-ema-uno` |

> **Si necesitas los logs de todos los firmware** (`test-nrf24-uno`, `test-laser-uno 10k LDR`, `test-ema-uno D2/D3`) dímelo y los vuelco a `docs/logs/firmware_sprint3/` + sección §6 con su `printDetails()` y `EMA α=0.2` correspondiente.

**Referencias:** `docs/prd.md:50 KPI <50ms` `docs/requirements.md:22 RF` `docs/architecture.md:60 topics` `docs/sprints.md:29 Sprint3` `docs/backlog.md:22 MOV-05/06` `docs/testing-rf-sprint1.md:32` `app/src/.../JoystickScreen.kt:1`
